In [ ]:
import os
from os.path import join
import pickle
from itertools import combinations
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

import mne
from mne import spatial_src_adjacency
from mne.stats import (
    spatio_temporal_cluster_test,
    summarize_clusters_stc,
    combine_adjacency,
    f_mway_rm,
    f_threshold_mway_rm
)

In [ ]:
## modify the test variables here
hemi = 'lh'
use_roi = True # subset stcs with spatial_exclude in test
search_tmin = 0
search_tmax = 0.4 # 400ms

In [ ]:
expt = 'EXPT'
ROOT = f'/path/to/{expt}'
os.chdir(ROOT)

subjects_dir = join(ROOT,'mri')
log_dir  = join(ROOT, 'logs')
stc_dir = join(ROOT, 'stc')
stats_dir = join(ROOT, 'stats')

excluded = ['R0000']
subjects = [i[:5] for i in os.listdir(subjects_dir) if i.startswith('R') and i[:5] not in excluded and not i.endswith(".fif")]

output_dir = os.path.join(ROOT, 'figures')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# brodmann areas of interest
brodmann_areas = ['20', '21', '22', '38', '39', '11', '44', '45']

# define colors for conditions
colors = {
    'CONDITION1': 'tab:blue',
    'CONDITION2': 'tab:orange',
    'CONDITION3': 'tab:green',
    'CONDITION4': 'tab:red'
}

conditions = list(colors.keys())

In [ ]:
def get_STC(subj, cond):
    stc_fname = os.path.join(stc_dir, '%s', '%s_%s_dSPM') % (cond, subj, cond)
    stc = mne.read_source_estimate(stc_fname, subject='fsaverage')
    return stc

In [ ]:
### set up ROI
# this gives you a Brodmann area label as a subset, but more options can be found in /mri/fsaverage/label
num = brodmann_areas[0]

## subset by brodmann (as an example):
annot_name = 'PALS_B12_Brodmann'
brodmann = mne.read_labels_from_annot('fsaverage', annot_name, subjects_dir = subjects_dir, hemi = hemi)
label_name = f'Brodmann.{num}-{hemi}'
roi = [label for label in brodmann if label.name==label_name][0] # you can also add labels for a larger roi

# look for vertex indices in the ROIs 
n_hemisources = 2562
hemi_idx      = np.arange(0, n_hemisources, 1)
roi_idx       = roi.get_vertices_used(vertices=hemi_idx)
diff_idx      = np.setdiff1d(hemi_idx, roi_idx)

spatial_exclude = diff_idx if use_roi else None

In [ ]:
# load in data and subset by search_tmin, search_tmax
data_time = []
data = []

for c, cond in enumerate(conditions):
    times = np.arange(-100, 801, 1)
    n_subj = len(subjects)
    data_tmp = np.empty((n_subj, n_hemisources, len(times)))
    print(f'Reading in STCs for condition {cond}')
    
    for s, subj in enumerate(subjects):
        stc = get_STC(subj, cond)
        data_tmp[s,:,:] = stc.data[:n_hemisources]
    
    data.append(np.transpose(data_tmp, [0, 2, 1]))
    
    tmin_idx = np.squeeze(np.where(times==search_tmin))
    tmax_idx = np.squeeze(np.where(times==search_tmax))
    toi_idx = np.arange(tmin_idx, tmax_idx+1)
    data_time.append(data[c][:,toi_idx,:n_hemisources])
    
print(f"Time window of analysis: {search_tmin} to {search_tmax}")

In [ ]:
## subset by brodmann (as an example):
for num in ['11', '38', '20', '21', '22', '39', '45']:
    annot_name = 'PALS_B12_Brodmann'
    brodmann = mne.read_labels_from_annot('fsaverage', annot_name, subjects_dir = subjects_dir, hemi = hemi)
    label_name = f'Brodmann.{num}-{hemi}'
    roi = [label for label in brodmann if label.name==label_name][0]

    # look for vertex indices in the ROIs 
    n_hemisources = 2562
    hemi_idx    = np.arange(0, n_hemisources, 1)
    roi_idx     = roi.get_vertices_used(vertices=hemi_idx)
    diff_idx    = np.setdiff1d(hemi_idx, roi_idx)

    spatial_exclude = diff_idx if use_roi else None

    src_fname = os.path.join(subjects_dir, 'fsaverage', 'bem', 'fsaverage-ico-4-src.fif')
    src = mne.read_source_spaces(src_fname)

    adjacency = None
    if hemi == 'lh':
        adjacency = mne.spatial_src_adjacency(src[:1]) # lh: src[:1], rh: src[1:]
    elif hemi == 'rh':
        adjacency = mne.spatial_src_adjacency(src[1:]) # lh: src[:1], rh: src[1:]
    else:
        adjacency = mne.spatial_src_adjacency(src)

    ttest = 'ttest'
    X_mne = data_time

    p_threshold = 0.001  # Cluster-forming threshold (strict)
    df = len(subjects) - 1 
    t_threshold = stats.distributions.t.ppf(1 - p_threshold / 2, df=df)
    print(f"Using t-threshold: {t_threshold} (p={p_threshold})")

    #Run the permutation cluster test for each pair
    for i, y in combinations(conditions, 2):
        if i != y:
            X = X_mne
            print(i, y)
            print("Conditions: ", conditions)
            print("Launching clustering test for test:", ttest)
            t_obs, clusters, clusters_pval, H0 = clu = mne.stats.spatio_temporal_cluster_test(
                    [X_mne[i], X_mne[y]],  # List of conditions
                    n_permutations=10000,
                    threshold=t_threshold,
                    tail=0,  # 0 for two-tailed test
                    adjacency=adjacency,
                    n_jobs=-1,
                    spatial_exclude=spatial_exclude
                )

            # Save results
            pickle_fname = os.path.join(stats_dir, 
                                        'stats_%s_%s_%s-%s_%s.pickled' 
                                        % (ttest, f"{i}_{y}", str(search_tmin), str(search_tmax), label_name))
            with open(pickle_fname, "wb") as open_file:
                pickle.dump(clu, open_file)

            p_thresh = 0.1
            print(clusters_pval)
            print(np.where(clusters_pval < p_thresh)[0])
            good_cluster_inds = np.where(clusters_pval < p_thresh)[0]
